### Exercise 2 - Proof of Work: Mining, Difficulty and Probability

In [7]:
# Import SHA-256, timing, and table-processing libraries.
import hashlib
import time
import pandas as pd

def mine_block(block_data, difficulty):
    # A valid hash must begin with the required number of zeros.
    target_prefix = "0" * difficulty
    nonce = 0
    attempts = 0
    start = time.perf_counter()

    # Try different nonces until a valid hash is found.
    while True:
        candidate = f"{block_data}|{nonce}"
        block_hash = hashlib.sha256(candidate.encode("utf-8")).hexdigest()
        attempts += 1

        if block_hash.startswith(target_prefix):
            elapsed = time.perf_counter() - start
            return nonce, block_hash, attempts, elapsed

        nonce += 1

# Test four difficulty levels with 30 blocks each.
difficulties = [2, 3, 4, 5]
runs_per_difficulty = 30
records = []

for difficulty in difficulties:
    print(f"\n{'='*25} DIFFICULTY {difficulty} {'='*25}")
    for run in range(1, runs_per_difficulty + 1):
        # Use unique data so each mining run has a new search problem.
        block_data = (
            f"COMP842|difficulty={difficulty}|run={run}|"
            f"unique={time.time_ns()}"
        )

        nonce, valid_hash, attempts, elapsed = mine_block(
            block_data, difficulty
        )

        # Store the measurements needed for the performance analysis.
        records.append({
            "Difficulty": difficulty,
            "Run": run,
            "Nonce": nonce,
            "Hash Attempts": attempts,
            "Mining Time (s)": elapsed,
            "Valid Hash": valid_hash,
        })

        print(
            f"Run {run:02d} | nonce={nonce:<10} | "
            f"attempts={attempts:<10} | time={elapsed:.6f}s | "
            f"hash={valid_hash}"
        )

# Convert all 120 mining runs into a DataFrame.
runs_df = pd.DataFrame(records)




========================= DIFFICULTY 2 =========================
Run 01 | nonce=316        | attempts=317        | time=0.000280s | hash=00cc509e9d9ec8d8b699c2a60be28ed81ac67b492aa34fe0b58bc76afba62ca3
Run 02 | nonce=179        | attempts=180        | time=0.000225s | hash=006432bbe7aff26304dc531ce8762018b9034b54375300f2e94b2842a8514526
Run 03 | nonce=14         | attempts=15         | time=0.000016s | hash=006a4a9a821b78ada6b891e9130e49048dbdc1ff680cad3b3f92abbfd6a09827
Run 04 | nonce=255        | attempts=256        | time=0.000230s | hash=000732320eb2974fea19e14c3f8ed3f3dc86c0b593785dab1355122ab8c8a37e
Run 05 | nonce=118        | attempts=119        | time=0.000109s | hash=00aae579c90e4eeba3953f731f9845585ee0cb7973d347dde4d7c2bb659ad70d
Run 06 | nonce=131        | attempts=132        | time=0.000111s | hash=0017d6cccea96df452203255162322b8c3e20a46d1aa699cefa8502fe2485695
Run 07 | nonce=282        | attempts=283        | time=0.000220s | hash=0027615e3642ec70efe9c33f5a9178b2944b409a

In [8]:
# Calculate the required performance metrics for each difficulty.
summary_rows = []

for difficulty in difficulties:
    subset = runs_df[runs_df["Difficulty"] == difficulty]

    measured_average_attempts = subset["Hash Attempts"].mean()
    theoretical_attempts = 16 ** difficulty
    total_attempts = subset["Hash Attempts"].sum()
    total_time = subset["Mining Time (s)"].sum()

    summary_rows.append({
        "Difficulty": difficulty,
        "Blocks Mined": len(subset),
        "Average Mining Time (s)": subset["Mining Time (s)"].mean(),
        "Minimum Mining Time (s)": subset["Mining Time (s)"].min(),
        "Maximum Mining Time (s)": subset["Mining Time (s)"].max(),
        "Std Dev Mining Time (s)": subset["Mining Time (s)"].std(ddof=1),
        "Mining Throughput (hashes/s)": total_attempts / total_time,
        "Measured Avg Attempts": measured_average_attempts,
        "Theoretical Attempts (16^d)": theoretical_attempts,
        "Measured/Theoretical": measured_average_attempts / theoretical_attempts,
        "Example Valid Hash": subset.iloc[0]["Valid Hash"],
    })

# Create the complete summary DataFrame.
summary_df = pd.DataFrame(summary_rows)

# Display metrics vertically so all required columns fit on screen.
summary_display = (
    summary_df.drop(columns=["Example Valid Hash"])
    .set_index("Difficulty")
    .T
)

print("\n=== REQUIRED PERFORMANCE SUMMARY ===")
print(summary_display.to_string(float_format=lambda value: f"{value:.6f}"))

print("\n=== EXAMPLE VALID HASHES ===")
print(summary_df[["Difficulty", "Example Valid Hash"]].to_string(index=False))



=== REQUIRED PERFORMANCE SUMMARY ===
Difficulty                               2              3              4              5
Blocks Mined                     30.000000      30.000000      30.000000      30.000000
Average Mining Time (s)           0.000349       0.001802       0.024169       0.447666
Minimum Mining Time (s)           0.000004       0.000020       0.000028       0.008751
Maximum Mining Time (s)           0.002499       0.011880       0.164852       2.303779
Std Dev Mining Time (s)           0.000467       0.002538       0.036151       0.492729
Mining Throughput (hashes/s) 660546.886357 2065021.546582 2707128.419905 2673625.987382
Measured Avg Attempts           230.266667    3720.566667   65429.233333 1196891.766667
Theoretical Attempts (16^d)     256.000000    4096.000000   65536.000000 1048576.000000
Measured/Theoretical              0.899479       0.908341       0.998371       1.141445

=== EXAMPLE VALID HASHES ===
 Difficulty                                         

In [11]:
# Compare measured growth with the theoretical 16x increase per zero.
growth_rows = []
for previous_difficulty, current_difficulty in zip(difficulties[:-1], difficulties[1:]):
    previous = summary_df[summary_df["Difficulty"] == previous_difficulty].iloc[0]
    current = summary_df[summary_df["Difficulty"] == current_difficulty].iloc[0]

    growth_rows.append({
        "Difficulty Step": f"{previous_difficulty} -> {current_difficulty}",
        "Average Time Ratio": (
            current["Average Mining Time (s)"] /
            previous["Average Mining Time (s)"]
        ),
        "Average Attempts Ratio": (
            current["Measured Avg Attempts"] /
            previous["Measured Avg Attempts"]
        ),
        "Theoretical Ratio": 16.0,
    })

growth_df = pd.DataFrame(growth_rows)
print("\n=== GROWTH BETWEEN DIFFICULTY LEVELS ===")
print(growth_df.to_string(index=False))

# Confirm every result satisfies its requested difficulty.
runs_df["Hash Valid"] = runs_df.apply(
    lambda row: row["Valid Hash"].startswith("0" * int(row["Difficulty"])),
    axis=1
)
print("\nAll 120 hashes satisfy their difficulty:", runs_df["Hash Valid"].all())



=== GROWTH BETWEEN DIFFICULTY LEVELS ===
Difficulty Step  Average Time Ratio  Average Attempts Ratio  Theoretical Ratio
         2 -> 3            5.168411               16.157643               16.0
         3 -> 4           13.414623               17.585825               16.0
         4 -> 5           18.522144               18.292921               16.0

All 120 hashes satisfy their difficulty: True
